# Volatility Smile Calibration

This notebook demonstrates volatility smile calibration to market data.

## Topics Covered:
1. Implied Volatility Calculation
2. Volatility Smile Pattern
3. Parametric Model Fitting (Quadratic, SVI, Polynomial)
4. Model Comparison
5. Volatility Surface Construction

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
import sys
sys.path.insert(0, '../src')

from mc_pricing import BlackScholes, Visualizer
from mc_pricing.calibration import SmileCalibration, VolatilitySurface
from mc_pricing.utils import DataLoader

%matplotlib inline
plt.style.use('seaborn-v0_8')

## 1. Generate Synthetic Market Data

Create realistic option prices with volatility smile.

In [ ]:
np.random.seed(42)

# Market parameters
spot = 100.0
maturity = 1.0
rate = 0.05

# Generate strikes
strikes = np.array([80, 85, 90, 95, 100, 105, 110, 115, 120])

# True volatilities with smile (quadratic pattern)
# Volatility smile: higher vol for OTM options
true_vols = []
for K in strikes:
    moneyness = K / spot
    # Create U-shaped smile
    smile = 0.20 + 0.10 * (moneyness - 1.0) ** 2
    true_vols.append(smile)

true_vols = np.array(true_vols)

# Generate market prices from true volatilities
market_prices = []
for K, vol in zip(strikes, true_vols):
    price = BlackScholes.price('call', spot, K, maturity, rate, vol)
    # Add small noise
    noise = np.random.normal(0, 0.05)
    market_prices.append(max(price + noise, 0.01))

market_prices = np.array(market_prices)

# Display market data
print("Synthetic Market Data:")
print(f"{'Strike':<10} {'Price':<12} {'True Vol':<12} {'Moneyness':<12}")
print("-" * 50)
for K, P, vol in zip(strikes, market_prices, true_vols):
    moneyness = K / spot
    print(f"{K:<10.0f} ${P:<11.2f} {vol*100:<11.1f}% {moneyness:<12.2f}")

## 2. Calculate Implied Volatilities

In [ ]:
# Calculate implied volatilities from market prices
implied_vols = []

for K, price in zip(strikes, market_prices):
    iv = BlackScholes.implied_volatility(
        option_type='call',
        market_price=price,
        spot=spot,
        strike=K,
        maturity=maturity,
        rate=rate
    )
    implied_vols.append(iv)

implied_vols = np.array(implied_vols)

# Plot the smile
moneyness = strikes / spot

plt.figure(figsize=(10, 6))
plt.plot(moneyness, true_vols * 100, 'o-', label='True Volatility', 
         markersize=10, linewidth=2.5, color='blue')
plt.plot(moneyness, implied_vols * 100, 's--', label='Implied from Prices', 
         markersize=8, linewidth=2, color='red')
plt.axvline(x=1.0, color='gray', linestyle=':', alpha=0.7, label='ATM')
plt.xlabel('Moneyness (K/S)', fontsize=12)
plt.ylabel('Implied Volatility (%)', fontsize=12)
plt.title('Volatility Smile', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nImplied Volatility Pattern:")
print("- Lowest vol at-the-money (ATM)")
print("- Higher vol for out-of-the-money (OTM) options")
print("- Classic 'smile' or 'smirk' shape")

## 3. Calibrate Parametric Models

### 3.1 Quadratic Model

In [ ]:
# Fit quadratic model: σ(K) = a + b*(K-K0) + c*(K-K0)^2
quad_cal = SmileCalibration(model='quadratic')
quad_result = quad_cal.fit(
    strikes=strikes,
    market_prices=market_prices,
    spot=spot,
    maturity=maturity,
    rate=rate,
    option_type='call'
)

print("Quadratic Model Calibration:")
print(f"  Parameters: a={quad_result['params']['a']:.6f}, "
      f"b={quad_result['params']['b']:.6f}, c={quad_result['params']['c']:.6f}")
print(f"  RMSE: {quad_result['rmse']:.6f}")
print(f"  Max Error: {quad_result['max_error']:.6f}")
print(f"  Points fitted: {quad_result['n_points']}")

### 3.2 SVI Model

In [ ]:
# Fit SVI model
svi_cal = SmileCalibration(model='svi')
svi_result = svi_cal.fit(
    strikes=strikes,
    market_prices=market_prices,
    spot=spot,
    maturity=maturity,
    rate=rate,
    option_type='call'
)

print("\nSVI Model Calibration:")
print(f"  Parameters:")
for key, val in svi_result['params'].items():
    print(f"    {key}: {val:.6f}")
print(f"  RMSE: {svi_result['rmse']:.6f}")
print(f"  Max Error: {svi_result['max_error']:.6f}")

### 3.3 Polynomial Model

In [ ]:
# Fit polynomial model
poly_cal = SmileCalibration(model='polynomial')
poly_result = poly_cal.fit(
    strikes=strikes,
    market_prices=market_prices,
    spot=spot,
    maturity=maturity,
    rate=rate,
    option_type='call'
)

print("\nPolynomial Model Calibration:")
print(f"  Coefficients: {poly_result['params']['coeffs']}")
print(f"  Degree: {poly_result['params']['degree']}")
print(f"  RMSE: {poly_result['rmse']:.6f}")
print(f"  Max Error: {poly_result['max_error']:.6f}")

## 4. Compare Model Fits

In [ ]:
# Generate fitted curves
strike_fine = np.linspace(strikes.min(), strikes.max(), 100)
moneyness_fine = strike_fine / spot

quad_vols = quad_cal.volatility(strike_fine)
svi_vols = svi_cal.volatility(strike_fine)
poly_vols = poly_cal.volatility(strike_fine)

# Plot comparison
plt.figure(figsize=(12, 7))

# Market implied vols
plt.plot(moneyness, implied_vols * 100, 'ko', markersize=10, 
         label='Market Implied Vol', zorder=5)

# Fitted models
plt.plot(moneyness_fine, quad_vols * 100, '-', linewidth=2.5, 
         label=f'Quadratic (RMSE: {quad_result["rmse"]:.4f})')
plt.plot(moneyness_fine, svi_vols * 100, '--', linewidth=2.5, 
         label=f'SVI (RMSE: {svi_result["rmse"]:.4f})')
plt.plot(moneyness_fine, poly_vols * 100, ':', linewidth=2.5, 
         label=f'Polynomial (RMSE: {poly_result["rmse"]:.4f})')

plt.axvline(x=1.0, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('Moneyness (K/S)', fontsize=12)
plt.ylabel('Implied Volatility (%)', fontsize=12)
plt.title('Volatility Smile Model Comparison', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Comparison table
comparison = pd.DataFrame({
    'Model': ['Quadratic', 'SVI', 'Polynomial'],
    'RMSE': [quad_result['rmse'], svi_result['rmse'], poly_result['rmse']],
    'Max Error': [quad_result['max_error'], svi_result['max_error'], poly_result['max_error']],
    'Parameters': [3, 5, 5]
})

print("\nModel Comparison:")
print(comparison.to_string(index=False))
print(f"\nBest Model (by RMSE): {comparison.loc[comparison['RMSE'].idxmin(), 'Model']}")

## 5. Fit Quality Analysis

In [ ]:
# Calculate fitted vols at market strikes
quad_fitted = quad_cal.volatility(strikes)
svi_fitted = svi_cal.volatility(strikes)
poly_fitted = poly_cal.volatility(strikes)

# Error analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = [
    ('Quadratic', quad_fitted),
    ('SVI', svi_fitted),
    ('Polynomial', poly_fitted)
]

for ax, (name, fitted) in zip(axes, models):
    errors = (fitted - implied_vols) * 100  # In basis points
    
    ax.bar(moneyness, errors, color=['red' if e < 0 else 'green' for e in errors],
           alpha=0.7, edgecolor='black')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax.set_xlabel('Moneyness (K/S)', fontsize=11)
    ax.set_ylabel('Error (%)', fontsize=11)
    ax.set_title(f'{name} Fitting Errors', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add RMSE
    rmse = np.sqrt(np.mean(errors**2))
    ax.text(0.02, 0.98, f'RMSE: {rmse:.3f}%', 
            transform=ax.transAxes, va='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## 6. Build Volatility Surface

Extend smile to multiple maturities.

In [ ]:
# Load or generate volatility surface data
try:
    # Try to load sample surface
    strikes_surf, maturities_surf, vols_surf = DataLoader.load_volatility_surface(
        '../data/market_data/sample_vol_surface.csv'
    )
    print("Loaded volatility surface from file")
except:
    # Generate sample surface
    strikes_surf, maturities_surf, vols_surf = DataLoader.generate_sample_volatility_surface(
        spot=spot,
        strikes_pct=np.linspace(0.8, 1.2, 11),
        maturities=np.array([0.25, 0.5, 1.0, 2.0]),
        atm_vol=0.2,
        smile_amplitude=0.05
    )
    print("Generated synthetic volatility surface")

print(f"\nSurface dimensions:")
print(f"  Strikes: {len(strikes_surf)}")
print(f"  Maturities: {len(maturities_surf)}")
print(f"  Total points: {vols_surf.size}")

In [ ]:
# Create VolatilitySurface object
vol_surface = VolatilitySurface()
vol_surface.fit(strikes_surf, maturities_surf, vols_surf, spot=spot)

# Visualize 3D surface
viz = Visualizer()
viz.plot_volatility_surface(
    strikes_surf,
    maturities_surf,
    vols_surf,
    spot=spot,
    title="Implied Volatility Surface"
)

## 7. Query Volatility Surface

In [ ]:
# Query specific points
test_queries = [
    (95, 0.5),
    (100, 1.0),
    (105, 1.5),
    (110, 0.75)
]

print("Volatility Surface Queries:")
print(f"{'Strike':<10} {'Maturity':<12} {'Volatility':<12} {'Moneyness':<12}")
print("-" * 50)

for strike, maturity in test_queries:
    vol = vol_surface.volatility(strike, maturity)
    moneyness = strike / spot
    print(f"{strike:<10.0f} {maturity:<12.2f} {vol*100:<11.2f}% {moneyness:<12.2f}")

# Get ATM term structure
print("\nATM Volatility Term Structure:")
for T in [0.25, 0.5, 1.0, 1.5, 2.0]:
    atm_vol = vol_surface.atm_volatility(T)
    print(f"  {T:4.2f}Y: {atm_vol*100:5.2f}%")

## 8. Smile Evolution with Maturity

In [ ]:
# Plot smiles for different maturities
fig, ax = plt.subplots(figsize=(12, 7))

maturities_to_plot = [0.25, 0.5, 1.0, 2.0]
colors = plt.cm.viridis(np.linspace(0, 1, len(maturities_to_plot)))

for T, color in zip(maturities_to_plot, colors):
    moneyness_slice, vols_slice = vol_surface.moneyness_slice(
        maturity=T,
        moneyness_range=(0.85, 1.15),
        n_points=50
    )
    ax.plot(moneyness_slice, vols_slice * 100, linewidth=2.5, 
            label=f'T={T:.2f}Y', color=color)

ax.axvline(x=1.0, color='gray', linestyle='--', alpha=0.5, label='ATM')
ax.set_xlabel('Moneyness (K/S)', fontsize=12)
ax.set_ylabel('Implied Volatility (%)', fontsize=12)
ax.set_title('Volatility Smile Evolution Across Maturities', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Key Observations:")
print("- Smile flattens for longer maturities")
print("- Short-dated options show steeper smile")
print("- ATM vol increases with maturity (term structure)")

## Summary

### Key Takeaways:

1. **Volatility Smile**: Market implied vols vary with strike (not constant)
2. **Model Selection**: 
   - Quadratic: Simple, fast, good for symmetric smiles
   - SVI: Industry standard, flexible, no-arbitrage
   - Polynomial: Very flexible, may overfit
3. **Surface Construction**: Interpolate across strikes and maturities
4. **Applications**: Risk management, exotic option pricing, hedging

### Best Practices:
- Use SVI for production systems
- Validate no-arbitrage conditions
- Recalibrate frequently (daily for liquid markets)
- Monitor calibration quality (RMSE, max error)

Next: Check out `06_full_comparison.ipynb` for comprehensive Monte Carlo vs Black-Scholes comparison!